In [3]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np


In [5]:
df=pd.read_csv(r"C:\Users\TABISH AHMAD\Documents\DataSets\tweet_emotions.csv").drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [10]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def lemmatization(text: str) -> str:
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

def remove_stop_words(text: str) -> str:
    tokens = text.split()
    tokens = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(tokens)

def lower_case(text: str) -> str:
    tokens = text.split()
    tokens = [word.lower() for word in tokens]
    return " ".join(tokens)

def removing_punctuations(text: str) -> str:
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def removing_urls(text: str) -> str:
    urls_pattern = re.compile(r"https?://\S+|www\.\S+")
    return urls_pattern.sub(r"", text)

def remove_small_sentences(df: pd.DataFrame, colname: str) -> pd.DataFrame:
    df.loc[df[colname].str.split().str.len() < 3, colname] = np.nan
    return df

def normalize_text(df: pd.DataFrame, colname: str) -> pd.DataFrame:
    df[colname] = df[colname].astype(str)
    df[colname] = df[colname].apply(lower_case)
    df[colname] = df[colname].apply(removing_urls)
    df[colname] = df[colname].apply(removing_punctuations)
    df[colname] = df[colname].apply(remove_stop_words)
    df[colname] = df[colname].apply(lemmatization)
    return df

In [13]:
df=normalize_text(df,"content")
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [14]:
df.sentiment.value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [15]:
x=df['sentiment'].isin(['happiness','sadness'])
df=df[x]

In [16]:
df.sentiment.value_counts()

sentiment
happiness    5209
sadness      5165
Name: count, dtype: int64

In [17]:
df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})
df.head()

C:\Users\TABISH AHMAD\AppData\Local\Temp\ipykernel_27360\3663829330.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})


,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im thinking old friend want married damn...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [19]:
vectorizer=CountVectorizer(max_features=1000)
x=vectorizer.fit_transform(df['content'])
y=df['sentiment']

In [21]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)


In [22]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/ahmadtab235/mlops-mini-projects.mlflow')
dagshub.init(repo_owner='ahmadtab235', repo_name='mlops-mini-projects', mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

Accessing as ahmadtab235

Initialized MLflow to track repo "ahmadtab235/mlops-mini-projects"

Repository ahmadtab235/mlops-mini-projects initialized!

2026/04/15 23:34:41 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/a89384719d7844afaa05a0f6065a5d04', creation_time=1776276282937, experiment_id='0', last_update_time=1776276282937, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [26]:
with mlflow.start_run():
    mlflow.log_param("vectorizer","Bag of words")
    mlflow.log_param("num_features",1000)
    mlflow.log_param("test_size",0.20)

    #model building and training
    model=LogisticRegression()
    model.fit(x_train,y_train)

    #log model parameters
    mlflow.log_param("model","Logistic Regression")

    #model evaluation
    y_pred=model.predict(x_test)
    accuracy=accuracy_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    f1=f1_score(y_test,y_pred)

    #log evaluation metrics
    mlflow.log_metric("accuracy",accuracy)
    mlflow.log_metric("precision",precision)
    mlflow.log_metric("recall",recall)
    mlflow.log_metric("f1_score",f1)

    #log model
    #mlflow.sklearn.log_model(model,"model")
    mlflow.sklearn.log_model(model, artifact_path="model")


    #save and log the notebook
    import os 
    notebook_path="exp1_baseline_model.ipynb"
    os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
    mlflow.log_artifact(notebook_path)

    #print the result for verification
    print(f"Accuracy : {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall : {recall}")
    print(f"F1_score : {f1}")


2026/04/16 00:17:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 00:17:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy : 0.7730120481927711
Precision: 0.7645914396887159
Recall : 0.774384236453202
F1_score : 0.7694566813509545
🏃 View run righteous-mule-456 at: https://dagshub.com/ahmadtab235/mlops-mini-projects.mlflow/#/experiments/0/runs/4a00b1c149174360bf66baefedc9f5a8
🧪 View experiment at: https://dagshub.com/ahmadtab235/mlops-mini-projects.mlflow/#/experiments/0


In [ ]:
yy